# Deployment

In [ ]:
!pip install -qqq streamlit

In [9]:
import os
import streamlit as st

In [10]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [11]:
%pwd
# !pwd

'/content'

In [12]:
%cd "/content/drive/MyDrive/AI_DS_ML PROJECTS/BREAST CANCER WISCONSIN/"

# os.chdir("/content/drive/MyDrive/AI_DS_ML PROJECTS/BREAST CANCER WISCONSIN/")

/content/drive/MyDrive/AI_DS_ML PROJECTS/BREAST CANCER WISCONSIN


In [13]:
%ls -la
# !ls -la

total 625
drwx------  4 root root   4096 Oct 21 13:05  ./
drwx------ 54 root root   4096 Sep 12 05:46  ../
-rw-------  1 root root  14428 Oct 23 15:14  bcw_app.ipynb
-rw-------  1 root root 480267 Oct 23 15:09  bcw_model_training.ipynb
-rw-------  1 root root   1779 Oct 23 09:10  breast_cancer_model.pkl
-rw-------  1 root root    181 Oct 23 14:25 '.ignore_material - to del.gdoc'
drwx------  2 root root   4096 Oct 23 14:21  IMAGES/
drwx------  2 root root   4096 Oct 23 14:25  RAW_DATA/
-rw-------  1 root root    181 Oct 23 14:39 'ReadMe Material.gdoc'
-rw-------  1 root root 124103 Jun 20  2023  wdbc.data


In [14]:
# app.py
%%writefile app.py

# --------------------------------------------------
# 1. PAGE CONFIGURATION
# --------------------------------------------------

st.set_page_config(
    page_title="Breast Cancer Classification",
    page_icon="🎗️",
    layout="wide"
)


# --------------------------------------------------
# 2. LOAD THE TRAINED MODEL
# --------------------------------------------------

MODEL_PATH = os.path.join(
    os.path.dirname(os.path.abspath(__file__)),
    "breast_cancer_model.pkl"
)


@st.cache_resource
def load_model(path):
    return joblib.load(path)


st.title("🎗️ Breast Cancer Classification")
st.subheader("PySpark + Machine Learning + Streamlit")

st.markdown(
    """
    This application demonstrates how a Logistic Regression model
    trained with PySpark can be used to classify the UCI Breast Cancer
    Wisconsin (Diagnostic) dataset.

    **Target classes:** Benign and Malignant.
    """
)

st.warning(
    "Educational demonstration only. This model is not clinically "
    "validated and must not be used to make medical decisions."
)

try:
    model = load_model(MODEL_PATH)
except Exception as exc:
    st.error(
        "Could not load the model. Ensure that "
        "`breast_cancer_model.pkl` exists in the project directory "
        "and is a trusted artifact created by your Colab notebook."
    )
    st.stop()


# --------------------------------------------------
# 3. EXTRACT MODEL PARAMETERS
# --------------------------------------------------

feature_names = model["feature_names"]

scaler_mean = np.asarray(model["scaler_mean"], dtype=float)
scaler_std = np.asarray(model["scaler_std"], dtype=float)
coefficients = np.asarray(model["coefficients"], dtype=float)
intercept = float(model["intercept"])
threshold = float(model["threshold"])

# Avoid division by zero
scaler_std = np.where(scaler_std == 0, 1.0, scaler_std)


# --------------------------------------------------
# 4. PREDICTION FUNCTION
# --------------------------------------------------

def predict_breast_cancer(input_df):
    """
    Predict using the parameters exported from the fitted
    PySpark StandardScaler + LogisticRegression pipeline.

    Expected input: pandas DataFrame containing the 30 features.
    """

    # Ensure the training feature order is preserved
    X = input_df[feature_names].to_numpy(dtype=float)

    if not np.isfinite(X).all():
        raise ValueError(
            "Input contains missing or non-finite values."
        )

    # Apply the same scaling learned during PySpark training
    X_scaled = (X - scaler_mean) / scaler_std

    # Logistic Regression linear score
    scores = X_scaled @ coefficients + intercept

    # Convert scores to probabilities
    probabilities = 1.0 / (
        1.0 + np.exp(-np.clip(scores, -500, 500))
    )

    # Apply the saved classification threshold
    predictions = (probabilities >= threshold).astype(int)

    return predictions, probabilities


# --------------------------------------------------
# 5. INPUT INTERFACE
# --------------------------------------------------

st.divider()

st.header("Make a prediction")

manual_tab, csv_tab = st.tabs(
    ["Enter features manually", "Upload a CSV file"]
)


# --------------------------------------------------
# 6. MANUAL INPUT
# --------------------------------------------------

with manual_tab:

    st.write(
        "Enter the 30 numeric features produced by the UCI "
        "dataset's image-analysis process. Use values from a "
        "valid dataset record for a meaningful demonstration."
    )

    with st.form("prediction_form"):

        st.caption(
            "Features are grouped by statistic: mean, standard error "
            "and worst. All 30 fields are required."
        )

        input_values = {}

        for group_name, start_index in [
            ("Mean features", 0),
            ("Standard error features", 10),
            ("Worst features", 20),
        ]:

            st.markdown(f"#### {group_name}")

            cols = st.columns(3)

            for i in range(start_index, start_index + 10):

                feature = feature_names[i]

                with cols[(i - start_index) % 3]:
                    input_values[feature] = st.number_input(
                        feature.replace("_", " ").title(),
                        value=0.0,
                        format="%.6f",
                        key=f"manual_{feature}"
                    )

        submitted = st.form_submit_button(
            "Predict",
            type="primary",
            use_container_width=True
        )

    if submitted:

        input_df = pd.DataFrame(
            [input_values],
            columns=feature_names
        )

        try:
            predictions, probabilities = predict_breast_cancer(
                input_df
            )

            predicted_class = int(predictions[0])
            malignant_probability = float(probabilities[0])

            st.divider()
            st.subheader("Prediction result")

            col1, col2 = st.columns(2)

            with col1:
                if predicted_class == 1:
                    st.error("Model output: Malignant")
                else:
                    st.success("Model output: Benign")

            with col2:
                st.metric(
                    "Estimated malignant probability",
                    f"{malignant_probability:.2%}"
                )

            st.progress(malignant_probability)

            st.caption(
                "This probability is the model's estimated output, "
                "not a clinically validated estimate of an individual's "
                "actual cancer risk."
            )

        except Exception as exc:
            st.error(f"Prediction failed: {exc}")


# --------------------------------------------------
# 7. CSV BATCH PREDICTIONS
# --------------------------------------------------

with csv_tab:

    st.write(
        "Upload a CSV containing the 30 feature columns. "
        "Column names must match the feature names used during training."
    )

    st.download_button(
        label="Download CSV template",
        data=pd.DataFrame(
            columns=feature_names
        ).to_csv(index=False),
        file_name="breast_cancer_input_template.csv",
        mime="text/csv"
    )

    uploaded_file = st.file_uploader(
        "Upload feature data",
        type=["csv"]
    )

    if uploaded_file is not None:

        try:
            input_df = pd.read_csv(uploaded_file)

            missing_columns = [
                col for col in feature_names
                if col not in input_df.columns
            ]

            if missing_columns:
                st.error(
                    "Missing required feature columns: "
                    + ", ".join(missing_columns)
                )

            elif input_df.empty:
                st.error("The uploaded CSV contains no records.")

            else:
                input_df = input_df[feature_names].apply(
                    pd.to_numeric,
                    errors="coerce"
                )

                if input_df.isna().any().any():
                    st.error(
                        "The CSV contains missing or non-numeric "
                        "feature values. Please correct the file."
                    )

                elif not np.isfinite(
                    input_df.to_numpy(dtype=float)
                ).all():
                    st.error(
                        "The CSV contains infinite or invalid values."
                    )

                else:
                    st.write("Input preview:")
                    st.dataframe(
                        input_df.head(),
                        use_container_width=True
                    )

                    if st.button(
                        "Run batch predictions",
                        type="primary"
                    ):

                        predictions, probabilities = (
                            predict_breast_cancer(input_df)
                        )

                        results_df = input_df.copy()

                        results_df["predicted_label"] = predictions

                        results_df["predicted_class"] = np.where(
                            predictions == 1,
                            "Malignant",
                            "Benign"
                        )

                        results_df[
                            "estimated_malignant_probability"
                        ] = probabilities

                        st.subheader("Batch prediction results")

                        st.dataframe(
                            results_df,
                            use_container_width=True
                        )

                        st.download_button(
                            label="Download predictions as CSV",
                            data=results_df.to_csv(index=False),
                            file_name="breast_cancer_predictions.csv",
                            mime="text/csv"
                        )

                        st.caption(
                            "Outputs are experimental model predictions "
                            "and are not medical diagnoses."
                        )

        except Exception as exc:
            st.error(f"Could not process the uploaded CSV: {exc}")


# --------------------------------------------------
# 8. FOOTER
# --------------------------------------------------

st.divider()

st.caption(
    "Dataset: UCI Breast Cancer Wisconsin (Diagnostic). "
    "Model: PySpark Logistic Regression. "
    "For educational and demonstration purposes only."
)

Writing app.py


In [15]:
%%writefile requirements.txt

streamlit
numpy
pandas
joblib

Writing requirements.txt


In [18]:
!pip install -qr requirements.txt

In [19]:
!pwd

/content/drive/MyDrive/AI_DS_ML PROJECTS/BREAST CANCER WISCONSIN


In [20]:
# start Streamlit
!streamlit run app.py --server.port 8501



2026-10-23 15:16:15.521 Uvicorn server started on :::8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://35.193.122.22:8501

  Stopping...
  Stopping...
